In [1]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

In [4]:
print("> Compute similarities")
print(f"\tv1 : {v1.dot(dv)} ")
print(f"\tv2 : {v2.dot(dv)} ")

> Compute similarities
	v1 : 0.3233238326017993 
	v2 : 0.019730417971579928 


In [6]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-29 15:15:49--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8000::154, 2606:50c0:8002::154, 2606:50c0:8003::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8000::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-29 15:15:50 (40.3 MB/s) - ‘ingest.py’ saved [888/888]



In [7]:
from ingest import load_faq_data

documents = load_faq_data()

In [8]:
texts = [doc["question"] + " " + doc["answer"] for doc in documents]

In [9]:
texts[:3]

["Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 "Course: What are the prerequisites for this course? To get the most out of this course, you should have:\n\n- Basic coding experience\n- Familiarity with SQL\n- Experience with Python (helpful but not required)\n\nNo prior data engineering experience is necessary. See [Readme on GitHub](https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/README.md#prerequisites).\n\nIf you have these basics, you're ready to start — you don't need to master everything up

In [11]:
texts.__len__()

1350

In [16]:
texts[49:50]

["My country isn't supported by GCP / my card is rejected. What can I do? GCP isn't available in some countries, and some cards are rejected even where it is. Workarounds students have used:\n\n- Try a different card. Cards from some banks (e.g. Kazakhstan-based Kaspi) sometimes don't work; cards from other banks/countries (e.g. TBC in Georgia) do.\n- Pyypl and similar virtual cards have worked for some.\n- If you can't get a GCP account at all, you can still complete most of the course locally — only Module 3's homework strictly requires BigQuery."]

In [24]:
from tqdm.auto import tqdm
import numpy as np

batch_size = 50
X = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = embed.encode_batch(batch)
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/27 [00:00<?, ?it/s]

In [20]:
query = "Can I still join the course after the start date?"
v_query = embed.encode(query)

scores = X.dot(v_query)
idx = np.argmax(scores)

In [22]:
print(f"\tscores : {scores} ")
print(f"\tidx : {idx} ")

	scores : [ 0.50192213  0.2651302   0.76294105 ... -0.08384743  0.03759797
 -0.12702167] 
	idx : 2 


In [23]:
documents[idx]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}